# Week 06 — Advanced Method Experiments

**Companion to:** Big Data and Efficient Processing  
**Course topic:** Machine Learning

This notebook includes **four executable micro-experiments** that deepen understanding of the weekly methods. Each example uses a minimal, inspectable computation. Every experiment asks you to change a parameter, observe the result, and explain the difference.

These are **teaching demonstrations** using simulated or toy data; they are not measured observations about an actual city.

## Data setup and libraries
The same `DATA_ONLY.zip` may be uploaded into a new Colab runtime. On local Jupyter, keep the data folder at the project root.

In [ ]:
from pathlib import Path
import io
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Local Jupyter: data/ or ../data. Colab: upload DATA_ONLY.zip once per runtime.
candidates = [Path('data'), Path('../data'), Path('/content/data')]
DATA = next((p for p in candidates if (p / 'cells.csv').is_file()), None)
if DATA is None:
    try:
        from google.colab import files
    except ImportError as exc:
        raise FileNotFoundError('Cannot find data/. Extract the course ZIP and keep the data folder with the notebooks.') from exc
    print('Select DATA_ONLY.zip from the downloaded course package:')
    uploaded = files.upload()
    zip_name = next((name for name in uploaded if name.endswith('.zip')), None)
    if zip_name is None:
        raise FileNotFoundError('Please upload DATA_ONLY.zip and run this cell again.')
    with zipfile.ZipFile(io.BytesIO(uploaded[zip_name])) as archive:
        archive.extractall(Path.cwd())
    DATA = Path('data')
assert (DATA / 'metadata.json').is_file(), 'Data package incomplete.'
print('Dataset folder:', DATA.resolve())
print('Synthetic teaching data: not field observations.')


## Experiment 1: Stream selected columns

**Purpose.** Streaming reads keep each memory chunk bounded.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import pandas as pd
sums, counts = {}, {}
for chunk in pd.read_csv(DATA/'benchmark_repeated_rows.csv', usecols=['date','lst_c'],chunksize=4000):
    g=chunk.groupby('date')['lst_c'].agg(['sum','count'])
    for dt,row in g.iterrows():
        sums[dt]=sums.get(dt,0)+row['sum']
        counts[dt]=counts.get(dt,0)+row['count']
print({k:sums[k]/counts[k] for k in sums})

### How to interpret this experiment

Streaming reads keep each memory chunk bounded.

- `pd.read_csv` — Reads CSV text into a pandas DataFrame. Inspect head() and column types after loading.
- `groupby` — Groups rows before counting or aggregating by sector, date, or other key.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Experiment 2: Weighted means of two chunks

**Purpose.** Pooling group means requires weighted sums and valid counts.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import numpy as np
chunk_means=np.array([30.,35.]); chunk_counts=np.array([2,8])
pooled=np.average(chunk_means,weights=chunk_counts)
assert pooled==34.
print(pooled)

### How to interpret this experiment

Pooling group means requires weighted sums and valid counts.

- `assert` — Checks a claimed condition and stops if it is false.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Experiment 3: Memory size of arrays

**Purpose.** Array memory increases with element count and dtype.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import numpy as np
x=np.zeros((500,500),dtype='float64')
print('Bytes:',x.nbytes,'MiB:',x.nbytes/1024**2)

### How to interpret this experiment

Array memory increases with element count and dtype.

- Read the code from top to bottom: identify inputs, transformations, outputs, and validation checks.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Experiment 4: Vectorized distances and memory

**Purpose.** Pairwise vectorization can create unexpectedly large temporary arrays.

**Scientific task:** Read the snippet, predict the output, run it, then explain the result and the assumptions.

In [ ]:
import numpy as np
A=np.array([[0.,0.],[100.,0.]])
B=np.array([[0.,100.],[200.,0.]])
D=np.sqrt(((A[:,None,:]-B[None,:,:])**2).sum(axis=2))
print(D)

### How to interpret this experiment

Pairwise vectorization can create unexpectedly large temporary arrays.

- Read the code from top to bottom: identify inputs, transformations, outputs, and validation checks.

**Questions:** (1) What does the printed output measure? (2) What happens if one input or parameter changes? (3) How could this method be extended to an independent geographic dataset?

## Reflection and extension

Compare the four experiments. Which operation depends on spatial location? Which depends on attribute values? Which requires an independent evaluation design? Record parameter values and assumptions.

**Suggested submission:** One figure or table, annotated Python code, and one paragraph comparing the changed setting to the baseline.